In [1]:
import ale_py
import torch
import gymnasium as gym
import random
from model import DeepQNetwork
from data import ReplayBuffer, Transition

gym.register_envs(ale_py)

In [2]:
num_frames = 4

def make_env(env_id: str):
    env = gym.make(env_id)
    env = gym.wrappers.GrayscaleObservation(env)
    env = gym.wrappers.FrameStackObservation(env, stack_size=num_frames)
    env = gym.wrappers.RecordEpisodeStatistics(env)    
    
    return env

env = make_env("ALE/SpaceInvaders-v5")
obs, info = env.reset()
obs = torch.from_numpy(obs).to(dtype=torch.float32).unsqueeze(0)

A.L.E: Arcade Learning Environment (version 0.11.2+ecc1138)
[Powered by Stella]


In [3]:
height = 210
width = 160
action_space = 6

# used to train to predict q-values
predictor_model = DeepQNetwork(
    img_height=210, 
    img_width=160, 
    action_space=6, 
    num_frames=4,
    kernel_size=4
).to('cuda')

# used to provide stable q-values for bellman equation.
target_model =  DeepQNetwork(
    img_height=210, 
    img_width=160, 
    action_space=6, 
    num_frames=4,
    kernel_size=4
).to('cuda')

target_model.load_state_dict(predictor_model.state_dict())
target_model.eval()

DeepQNetwork(
  (conv_layers): Sequential(
    (conv2d): Conv2d(4, 4, kernel_size=(4, 4), stride=(2, 2))
    (silu): SiLU()
    (conv2d2): Conv2d(4, 4, kernel_size=(4, 4), stride=(2, 2))
    (silu2): SiLU()
    (conv2d3): Conv2d(4, 1, kernel_size=(4, 4), stride=(2, 2))
    (silu3): SiLU()
  )
  (ff_layers): Sequential(
    (linear1): Linear(in_features=432, out_features=432, bias=True)
    (gelu): GELU(approximate='none')
    (linear2): Linear(in_features=432, out_features=6, bias=True)
  )
)

In [4]:
def select_action(state, epsilon):
    if random.random() < epsilon:
        return env.action_space.sample()
    else:
        return predictor_model(state).argmax(dim=1).item()

In [5]:
def train_step(
    replay_buffer: ReplayBuffer, 
    batch_size: int,
    optimizer,
    predictor_model,
    target_model,
    gamma,
    loss_func
) -> float:
    if len(replay_buffer) < batch_size:
        return

    states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)
    
    q_values = predictor_model(states).gather(1, actions.unsqueeze(1)).squeeze(1)

    next_q_values = target_model(next_states).max(dim=1).values

    # Bellman equation.
    target = rewards + gamma * next_q_values * (1 - dones)

    loss = loss_func(q_values, target.detach())

    optimizer.zero_grad() # clear grads

    loss.backward() # calc grads.

    optimizer.step() # update model

    return loss.item()

In [ ]:
# Reset env
episodes = 10
step_counter = 0
terminated = False
replay_buffer = ReplayBuffer(capacity=10000)

from torch.optim import AdamW

gamma = 0.99
epsilon = 1.0
epsilon_end = 0.01
epsilon_decay = 0.995
batch_size = 64
target_update_freq = 50
learning_rate = 1e-4

optimizer = AdamW(params=predictor_model.parameters(), lr=learning_rate)
loss_func = torch.nn.MSELoss()

for episode in range(episodes):
    state, info = env.reset()
    # states -> ( frames, height, width)
    state = torch.from_numpy(state).to(dtype=torch.float32, device='cuda')
    terminated = False
    truncated = False

    while not terminated and not truncated:
        action = select_action(state.unsqueeze(0), epsilon=epsilon)
        next_state, reward, terminated, truncated, info = env.step(action)
        
        # Prep the observation for the model
        next_state = torch.from_numpy(next_state).to(dtype=torch.float32, device='cuda')
        
        # Push to ReplayBuffer
        replay_buffer.push(Transition(state, action, reward, next_state, terminated))
        
        loss = train_step(
            replay_buffer=replay_buffer, 
            batch_size=batch_size,
            optimizer=optimizer,
            predictor_model=predictor_model,
            target_model=target_model,
            gamma=gamma,
            loss_func=loss_func
        )

        state = next_state
        step_counter += 1

        if step_counter % target_update_freq == 0:
            target_model.load_state_dict(predictor_model.state_dict())
    
    print(f"Episode: {episode}, loss: {loss}")

    epsilon = max(epsilon_end, epsilon*epsilon_decay)

    

Episode: 0, loss: 2.2662627696990967
Episode: 1, loss: 1.94629967212677
Episode: 2, loss: 2.06892728805542
Episode: 3, loss: 2.0793495178222656
Episode: 4, loss: 5.643962860107422
Episode: 5, loss: 2.151719331741333
Episode: 6, loss: 4.700164318084717
Episode: 7, loss: 2.4306163787841797
Episode: 8, loss: 2.4628045558929443
Episode: 9, loss: 4.914493560791016
